In [14]:
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials

# Define the scopes required to access Google Drive
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

# Authenticate using the service account credentials
creds = Credentials.from_service_account_file("credentials.json", scopes=SCOPES)

# Build the Google Drive service
service = build("drive", "v3", credentials=creds)

# Function to list all files in a specific folder and return them as a dictionary
def list_files_in_folder(folder_id):
    # Create a query to list files in the specified folder
    query = f"'{folder_id}' in parents"
    
    # Initialize the file list
    files = []
    page_token = None
    
    while True:
        # Make the API call to list files, handle pagination
        results = service.files().list(q=query, pageToken=page_token).execute()
        files.extend(results.get('files', []))
        
        # Check if there are more files to fetch
        page_token = results.get('nextPageToken', None)
        if not page_token:
            break

    if not files:
        print("No files found.")
        return {}
    else:
        # Create a dictionary with file names as keys and file IDs as values
        file_ids = {file['name']: file['id'] for file in files}
        
        # Optionally print the files in a nicely formatted way
        print(f"{'File Name':<50} {'File ID'}")
        print("="*80)
        for name, file_id in file_ids.items():
            print(f"{name:<50} {file_id}")
        
        # Return the dictionary
        return file_ids

# Replace with the actual folder ID
folder_id = "1glcAZMKgVHnh8JKcj27yLrohv9pKemTq"  # Replace with your folder ID

# List the files in the folder and store the results in file_ids
file_ids = list_files_in_folder(folder_id)

File Name                                          File ID
DOBAVA                                             1L8iyJAvrU_UxxgbKToI2GTs2gQl1ydE8L6Zjv_327mY
SETUP                                              1rjrYushy3HrFnzq5iu8Lg2BeX--Peewf
WINE_ORDER                                         1-9AOkPQmyHSiEzd3JEcn_rgADDKNMPqCXWrZJrVkGk8
CUSTOMERS                                          1usi3nFi4rIV6LsJm0TWeAOYkv15tWPsNYpuMQXQy5Xk
Avtomatizacija_Procesa_Narocil_Vinska_Klet.docx    1ioCIMzTmj3Q6TEVoVh3TYGg4QkT0vb8O
WINE_ORDERS_SUBMITED                               1Bmbxvi1ojCz1o_jJmR1K5lpl_9SBOXyBVxh4awpoWoI
VINO                                               1Co1NGIGHPHeLfQzLmtAvHAZ9905Zfnn1i6aJXWote-HOSe3eYDP6cf95


In [18]:
print(file_ids['WINE_ORDERS_SUBMITED'])


1Bmbxvi1ojCz1o_jJmR1K5lpl_9SBOXyBVxh4awpoWoI


In [23]:
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials

# Define the scopes required to access Google Sheets
SCOPES = ["https://www.googleapis.com/auth/spreadsheets.readonly"]

# Authenticate using the service account credentials
creds = Credentials.from_service_account_file("credentials.json", scopes=SCOPES)

# Build the Google Sheets service
service = build("sheets", "v4", credentials=creds)

# Define the spreadsheet ID
WINE_ORDERS_FILE_ID = file_ids['WINE_ORDERS_SUBMITED']  # Replace with the actual ID of the WINE orders file

# Function to get the first sheet's name and read the first 5 rows
def get_first_n_rows(n=5):
    try:
        # Get the spreadsheet metadata to list all sheet names
        spreadsheet = service.spreadsheets().get(spreadsheetId=WINE_ORDERS_FILE_ID).execute()
        
        # Get the first sheet's name
        first_sheet_name = spreadsheet['sheets'][0]['properties']['title']
        
        # Define the range to fetch the first 'n' rows (starting from row 1 for header and row 2 onwards for data)
        range_ = f"{first_sheet_name}!A1:P{n+1}"  # Adjust the end row based on n

        # Read data from the first 'n' rows
        sheet = service.spreadsheets().values().get(
            spreadsheetId=WINE_ORDERS_FILE_ID,
            range=range_
        ).execute()

        values = sheet.get('values', [])

        if not values:
            print("No data found in the specified range.")
        else:
            # Print the header row first (column names)
            header = values[0]
            print(f"{' | '.join(header)}")
            print("="*80)
            
            # Print the first 'n' rows in a nice format
            for row in values[1:]:  # Start from 1 to skip header row
                # Print entire row, ensuring the values are printed even if there are missing cells
                print(f"{' | '.join(str(cell) if cell else '' for cell in row)}")
                
    except Exception as e:
        print(f"Error retrieving data: {e}")

# Call the function to print the first 5 rows
get_first_n_rows()



Časovni žig | Refošk 25L sod | Malvazija 25L | Stranka/kupec | Malvazija 25L sod | Opomba | Malvazija 1L | Refošk 1L | Stolpec 8 | DRUGO | MALVAZIJA | MERLOT | REFOŠK | SHIRAZ | FLORIS | ROSE
7. 2. 2025 9:51:15 | 2 | 2
7. 2. 2025 10:29:57 | 4 |  | Marko Kovač | 3 | Plačano | 6 | 12 |  | NE
7. 2. 2025 10:59:11 | 5 |  | John Smith | 2 |  |  |  |  | NE


In [27]:
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials
from datetime import datetime

# Define the scopes required to access Google Sheets with read and write permissions
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",  # Add write access
    "https://www.googleapis.com/auth/drive.readonly"  # Keep read-only access to Drive
]

# Authenticate using the service account credentials
creds = Credentials.from_service_account_file("credentials.json", scopes=SCOPES)

# Build the Google Sheets service
service = build("sheets", "v4", credentials=creds)

def add_order_numbers():
    try:
        # Get the spreadsheet data
        sheet = service.spreadsheets().values().get(spreadsheetId=WINE_ORDERS_FILE_ID, range="WINE_ORDERS_SUBMITED!A2:P").execute()
        rows = sheet.get('values', [])
        
        # Dictionary to track the order count per date
        order_count_by_date = {}

        # Iterate over the rows to generate and add order numbers
        for i, row in enumerate(rows):
            # Extract timestamp from column A (first column)
            timestamp = row[0] if row else ""
            
            # If there's a valid timestamp, generate the order number
            if timestamp:
                try:
                    date = datetime.strptime(timestamp.split()[0], "%d. %m. %Y").strftime("%Y%m%d")
                except ValueError:
                    date = None

                if date:
                    # Increment order number for the same date
                    if date not in order_count_by_date:
                        order_count_by_date[date] = 1
                    else:
                        order_count_by_date[date] += 1
                    
                    # Generate the order number in format yyyymmddNNN
                    order_number = f"{date}{order_count_by_date[date]:03d}"
                    
                    # Ensure that the row has 17 columns and add the order number to column Q (index 16)
                    if len(row) < 17:
                        row.extend([""] * (17 - len(row)))  # Ensure the row has enough columns
                    row[16] = order_number  # Update the order number in the Q column

        # Write the updated rows back to the sheet
        if rows:
            # Dynamically calculate the number of rows
            num_rows = len(rows)
            update_range = f"WINE_ORDERS_SUBMITED!A2:P{num_rows + 1}"  # Update the range to match the number of rows
            body = {"values": rows}
            service.spreadsheets().values().update(
                spreadsheetId=WINE_ORDERS_FILE_ID,
                range=update_range,
                valueInputOption="RAW",
                body=body
            ).execute()
            print("Order numbers added successfully.")
        else:
            print("No rows found in the sheet.")
    except Exception as e:
        print(f"Error adding order numbers: {e}")

# Call the function to add order numbers to the sheet
add_order_numbers()



Error adding order numbers: <HttpError 400 when requesting https://sheets.googleapis.com/v4/spreadsheets/1Bmbxvi1ojCz1o_jJmR1K5lpl_9SBOXyBVxh4awpoWoI/values/WINE_ORDERS_SUBMITED%21A2%3AP?alt=json returned "Unable to parse range: WINE_ORDERS_SUBMITED!A2:P". Details: "Unable to parse range: WINE_ORDERS_SUBMITED!A2:P">
